In [1]:
import pandas as pd

file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_clean.csv"
df = pd.read_csv(file)

print(df.shape)
print(df.dtypes)
print(df.head())

print("\nAny missing complaint_id?", df["complaint_id"].isna().sum())
print("Any missing date_received?", df["date_received"].isna().sum())

print("\nMax length checks:")
print("product:", df["product"].astype(str).str.len().max())
print("sub_product:", df["sub_product"].astype(str).str.len().max())
print("issue:", df["issue"].astype(str).str.len().max())
print("sub_issue:", df["sub_issue"].astype(str).str.len().max())
print("company:", df["company"].astype(str).str.len().max())
print("state:", df["state"].astype(str).str.len().max())
print("submitted_via:", df["submitted_via"].astype(str).str.len().max())
print("timely_response:", df["timely_response"].astype(str).str.len().max())
print("consumer_disputed:", df["consumer_disputed"].astype(str).str.len().max())

(100000, 12)
date_received         object
product               object
sub_product           object
issue                 object
sub_issue             object
complaint_text        object
company               object
state                 object
submitted_via         object
timely_response       object
consumer_disputed    float64
complaint_id           int64
dtype: object
  date_received                                            product  \
0    2022-11-16  Credit reporting, credit repair services, or o...   
1    2022-02-22  Credit reporting, credit repair services, or o...   
2    2023-12-12                                    Debt collection   
3    2021-11-02  Credit reporting, credit repair services, or o...   
4    2023-04-09  Credit reporting, credit repair services, or o...   

        sub_product                                              issue  \
0  Credit reporting               Incorrect information on your report   
1  Credit reporting  Problem with a credit reporting com

In [2]:
output_file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_sql_ready.csv"

# -------------------------
# 1. complaint_id
# -------------------------
df["complaint_id"] = pd.to_numeric(df["complaint_id"], errors="coerce")
df = df.dropna(subset=["complaint_id"])
df["complaint_id"] = df["complaint_id"].astype("int64")

# -------------------------
# 2. date_received
# -------------------------
df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce")
df = df.dropna(subset=["date_received"])
df["date_received"] = df["date_received"].dt.strftime("%Y-%m-%d")

# -------------------------
# 3. Fill missing complaint_text
# -------------------------
df["complaint_text"] = df["complaint_text"].fillna("").astype(str).str.strip()

# -------------------------
# 4. Clean consumer_disputed
# -------------------------
# Right now it looks like float because of NaN values.
# Convert to clean text labels for SQL.
df["consumer_disputed"] = df["consumer_disputed"].fillna("Unknown").astype(str).str.strip()

# Optional normalization if weird values appear
df["consumer_disputed"] = df["consumer_disputed"].replace({
    "nan": "Unknown",
    "NaN": "Unknown",
    "": "Unknown"
})

# -------------------------
# 5. Clean other text columns
# -------------------------
text_cols = [
    "product",
    "sub_product",
    "issue",
    "sub_issue",
    "company",
    "state",
    "submitted_via",
    "timely_response"
]

for col in text_cols:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()

# -------------------------
# 6. Standardize timely_response
# -------------------------
df["timely_response"] = df["timely_response"].replace({
    "nan": "Unknown",
    "NaN": "Unknown",
    "": "Unknown"
})

# -------------------------
# 7. Trim to SQL-safe lengths
# -------------------------
df["product"] = df["product"].str[:255]
df["sub_product"] = df["sub_product"].str[:255]
df["issue"] = df["issue"].str[:255]
df["sub_issue"] = df["sub_issue"].str[:255]
df["company"] = df["company"].str[:255]
df["state"] = df["state"].str[:50]
df["submitted_via"] = df["submitted_via"].str[:100]
df["timely_response"] = df["timely_response"].str[:20]
df["consumer_disputed"] = df["consumer_disputed"].str[:20]

# complaint_text can be long; keep it as is for LONGTEXT
# But remove obvious null-like strings
df["complaint_text"] = df["complaint_text"].replace({
    "nan": "",
    "NaN": "",
    "None": "",
    "<NA>": "",
    "N/A": ""
})

# -------------------------
# 8. Remove duplicate complaint_id
# -------------------------
df = df.drop_duplicates(subset=["complaint_id"])

# -------------------------
# 9. Save SQL-ready file
# -------------------------
df.to_csv(output_file, index=False)

print("SQL-ready file created successfully.")
print("Final shape:", df.shape)
print("Saved to:", output_file)

print("\nData types:")
print(df.dtypes)

print("\nMissing complaint_id:", df["complaint_id"].isna().sum())
print("Missing date_received:", pd.to_datetime(df["date_received"], errors="coerce").isna().sum())
print("Missing complaint_text:", df["complaint_text"].isna().sum())

print("\nUnique timely_response values:")
print(df["timely_response"].value_counts(dropna=False).head(10))

print("\nUnique consumer_disputed values:")
print(df["consumer_disputed"].value_counts(dropna=False).head(10))

SQL-ready file created successfully.
Final shape: (100000, 12)
Saved to: C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_sql_ready.csv

Data types:
date_received        object
product              object
sub_product          object
issue                object
sub_issue            object
complaint_text       object
company              object
state                object
submitted_via        object
timely_response      object
consumer_disputed    object
complaint_id          int64
dtype: object

Missing complaint_id: 0
Missing date_received: 0
Missing complaint_text: 0

Unique timely_response values:
timely_response
Yes    98997
No      1003
Name: count, dtype: int64

Unique consumer_disputed values:
consumer_disputed
Unknown    100000
Name: count, dtype: int64


In [3]:
file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_sql_ready.csv"

df = pd.read_csv(file, low_memory=False)

print("Total rows:", len(df))
print("Unique complaint_id:", df["complaint_id"].nunique())
print("Duplicate count:", df.duplicated(subset=["complaint_id"]).sum())

dups = df[df.duplicated(subset=["complaint_id"], keep=False)].sort_values("complaint_id")
print(dups[["complaint_id"]].head(20))

Total rows: 100000
Unique complaint_id: 100000
Duplicate count: 0
Empty DataFrame
Columns: [complaint_id]
Index: []


In [4]:
import pandas as pd

input_file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_sql_ready.csv"
output_file = r"C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_mysql_safe.csv"

df = pd.read_csv(input_file, low_memory=False)

# Ensure key types
df["complaint_id"] = pd.to_numeric(df["complaint_id"], errors="coerce")
df = df.dropna(subset=["complaint_id"])
df["complaint_id"] = df["complaint_id"].astype("int64")

df["date_received"] = pd.to_datetime(df["date_received"], errors="coerce")
df = df.dropna(subset=["date_received"])
df["date_received"] = df["date_received"].dt.strftime("%Y-%m-%d")

# Fill and clean text columns
text_cols = [
    "product", "sub_product", "issue", "sub_issue", "complaint_text",
    "company", "state", "submitted_via", "timely_response", "consumer_disputed"
]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str).str.strip()

# Remove line breaks / tabs from complaint_text
df["complaint_text"] = (
    df["complaint_text"]
    .str.replace("\r", " ", regex=False)
    .str.replace("\n", " ", regex=False)
    .str.replace("\t", " ", regex=False)
)

# Replace repeated spaces
df["complaint_text"] = df["complaint_text"].str.replace(r"\s+", " ", regex=True)

# Clean obvious null-like values
df["consumer_disputed"] = df["consumer_disputed"].replace({
    "nan": "Unknown",
    "NaN": "Unknown",
    "": "Unknown"
})

df["timely_response"] = df["timely_response"].replace({
    "nan": "Unknown",
    "NaN": "Unknown",
    "": "Unknown"
})

# Trim lengths to match SQL schema safely
df["product"] = df["product"].str[:255]
df["sub_product"] = df["sub_product"].str[:255]
df["issue"] = df["issue"].str[:255]
df["sub_issue"] = df["sub_issue"].str[:255]
df["company"] = df["company"].str[:255]
df["state"] = df["state"].str[:50]
df["submitted_via"] = df["submitted_via"].str[:100]
df["timely_response"] = df["timely_response"].str[:20]
df["consumer_disputed"] = df["consumer_disputed"].str[:20]

# Remove duplicate IDs
df = df.drop_duplicates(subset=["complaint_id"])

# Save MySQL-safe CSV
df.to_csv(output_file, index=False, encoding="utf-8", quotechar='"')

print("MySQL-safe file created:")
print(output_file)
print("Final shape:", df.shape)

MySQL-safe file created:
C:\Users\ujjwa\Downloads\complaint-intelligence-platform\data\processed\complaints_mysql_safe.csv
Final shape: (100000, 12)
